In [ ]:
# --- parameters (patch_notebook_params.py) ---
MAX_EPOCHS = 2000
N_SAMPLES = 10           # run_metrics.py forces 50 for generative methods
RESET_TRAINING = False
CUDA_VISIBLE_DEVICES = "0"
METRICS_CSV = "results/metrics.csv"
SKIP_TRAINING = False    # run_metrics.py sets this True: load weights from
                         # the checkpoint directly instead of calling
                         # trainer.fit(), which can silently retrain for the
                         # full schedule if the checkpoint does not cleanly
                         # resume to exactly MAX_EPOCHS.


In [ ]:
# --- epoch heartbeat (patch_notebook_params.py) ---
import pytorch_lightning as _pl

class EpochHeartbeat(_pl.Callback):
    """Prints one clear progress line every `every_n_epochs` epochs, so
    sbatch logs show training progress without the noise of a per-batch
    tqdm progress bar (which doesn't render well once redirected to a
    plain log file).    """

    def __init__(self, every_n_epochs: int = 1):
        self.every_n_epochs = every_n_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1
        if epoch % self.every_n_epochs != 0 and epoch != trainer.max_epochs:
            return
        parts = []
        for k, v in sorted(trainer.callback_metrics.items()):
            try:
                parts.append(f'{k}={float(v):.4f}')
            except (TypeError, ValueError):
                pass
        print(f'[progress] epoch {epoch}/{trainer.max_epochs} | ' + ' | '.join(parts), flush=True)


## Flow Matching — Sea Ice Concentration (ASIP / OSISAF)

Conditional Flow Matching (Heun ODE) for SIC reconstruction.
- Target: ASIP SIC (9 time steps, 240×240) — last 9 of 15, contains NaN gaps
- Conditioning: gappy ASIP (input) + OSISAF coarse SIC
- Loss: computed only on non-NaN target pixels

In [ ]:
!nvidia-smi

In [ ]:
import os; os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

In [ ]:
import math
import os
import zipfile
import glob
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import xarray as xr
import pandas as pd
import torch
from torch import Tensor, nn
from pytorch_lightning import LightningModule, Trainer, seed_everything
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from matplotlib import pyplot as plt
from torchinfo import summary

import sys
sys.path.append('../..')      # -> consistency/ (flowmatching_models, spectral_utils)
sys.path.append('../../..')   # -> 4dvarnet-starter-devs/

from flowmatching_models.flowmatching_models_FM import (
    UNetFM, UNetFMConfig,
    LitFlowMatchingModel, LitFMConfig,
    LogScaleModel,
    heun_sample, euler_sample,
    neglogpdf, neglogcdf,
    sample_uniform_time, masked_average,
)

try:
    from properscoring import crps_ensemble
    HAS_PROPERSCORING = True
except ImportError:
    HAS_PROPERSCORING = False
    print("properscoring not installed — CRPS disabled.")

## DataModule — ASIP / OSISAF Sea Ice Concentration

- `batch.asip` (target): ASIP SIC, 9 time steps (last 9 of 15), 240×240 — has NaN gaps
- `batch.input`: gappy ASIP (sparse observations)
- `batch.osisaf`: OSISAF coarse SIC (conditioning)
- Conditioning = `cat(input, osisaf)` → 2C channels
- ASIP/input/osisaf are already a fraction in [0, 1] in the NetCDF file (same as
  the CM_sic notebook) — no rescaling needed. `norm_stats: [0, 1]` (identity).

In [ ]:
sys.path.insert(0, '../../..')
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import functools as ft
from src.dataloader_SIC import BaseDataModule, TrainingItem_4da

T_CROP = 9  # keep last 9 time steps out of 15

class CroppedDataModule(BaseDataModule):
    def __init__(self, *args, t_crop=9, **kwargs):
        super().__init__(*args, **kwargs)
        self.t_crop = t_crop

    def build_batch(self):
        # ASIP/input/osisaf are already in [0, 1] in the NetCDF file -- no
        # rescaling needed (same data source as CM_sic). The previous
        # `/ SIC_SCALE (100.0)` here was wrong: it assumed the raw values were
        # a percentage in [0, 100] (true for some other SIC datasets, not this
        # one), and crushed the target down to [0, 0.01] -- a 100x too-small
        # scale that still let the training loss decrease (the log-scale head
        # just learned a correspondingly tiny predicted std) while producing
        # near-noise output at sampling and a val RMSE that looked small in
        # absolute terms but was actually ~6x the natural scale of the
        # (wrongly shrunk) target.
        return ft.partial(ft.reduce, lambda i, f: f(i), [
            TrainingItem_4da._make,
        ])

    def setup(self, stage='test'):
        super().setup(stage)
        for ds in [self.train_ds, self.val_ds, self.test_ds]:
            ds.db = ds.db.isel(time=slice(-self.t_crop, None))

datamodule = CroppedDataModule(
    asip_paths="../../../data/asip_database_daw15_sparse.nc",
    split_train=slice(0, 300),
    split_val=slice(300, 350),
    split_test=slice(350, 393),
    da=True,
    norm_stats=[0, 1],
    norm_stats_covs=[
        {'t2m': 270.08, 'istl1': 267.68, 'siconc': 0, 'sst': 276.97, 'skt': 270.50},
        {'t2m': 14.67,  'istl1': 7.80,  'siconc': 1, 'sst': 6.82,   'skt': 15.21},
    ],
    t_crop=T_CROP,
)
datamodule.setup()

sample = datamodule.train_ds[0]
C = sample.asip.shape[0]
H, W = sample.asip.shape[1], sample.asip.shape[2]
print(f"C={C}, H={H}, W={W}")
print(f"Train: {len(datamodule.train_ds)}, Val: {len(datamodule.val_ds)}, Test: {len(datamodule.test_ds)}")
print(f"asip: {sample.asip.shape}  input: {sample.input.shape}  osisaf: {sample.osisaf.shape}")
print(f"asip range: [{np.nanmin(sample.asip):.3f}, {np.nanmax(sample.asip):.3f}]")
print(f"asip std: {np.nanstd(sample.asip):.4f}")
print(f"NaN fraction in asip: {np.isnan(sample.asip).mean()*100:.1f}%")

fig, axes = plt.subplots(3, min(C, 5), figsize=(3 * min(C, 5), 9))
for t_idx in range(min(C, 5)):
    axes[0, t_idx].imshow(sample.asip[t_idx], vmin=0, vmax=1, cmap='Blues_r')
    axes[0, t_idx].set_title(f't={t_idx}', fontsize=8); axes[0, t_idx].axis('off')
    axes[1, t_idx].imshow(sample.input[t_idx], vmin=0, vmax=1, cmap='Blues_r')
    axes[1, t_idx].axis('off')
    axes[2, t_idx].imshow(sample.osisaf[t_idx], vmin=0, vmax=1, cmap='Blues_r')
    axes[2, t_idx].axis('off')
axes[0, 0].set_ylabel('ASIP (target)', fontsize=10)
axes[1, 0].set_ylabel('Input (gappy)', fontsize=10)
axes[2, 0].set_ylabel('OSISAF (cond.)', fontsize=10)
plt.tight_layout()
plt.show()

### UNetFM — Velocity-prediction network

Input: `cat(x_t, y_cond_filled, mask_cond)` where `y_cond = cat(input, osisaf)` → C + 2×2C = 5C channels.

Uses `cond_channels=2*C` in UNetFMConfig to handle the wider conditioning.

In [ ]:
cfg = UNetFMConfig(channels=C, cond_channels=2*C)
net = UNetFM(cfg)

y_dummy = torch.cat([torch.randn(1, C, H, W), torch.randn(1, C, H, W)], dim=1)

summary(
    net,
    input_size=(
        (1, C, H, W),     # x_t
        (1, 2*C, H, W),   # y_cond = cat(input, osisaf)
        (1,),              # t
    ),
)

### LitFlowMatchingSIC — masked loss for gappy target

Overrides the training step to:
1. Build conditioning `y = cat(input, osisaf)`
2. Mask loss to non-NaN pixels in the ASIP target
3. Fill NaN in target before building the interpolant

In [ ]:
class LitFlowMatchingSIC(LitFlowMatchingModel):

    def training_step(self, batch, batch_idx: int):
        if isinstance(batch, list):
            batch = batch[0]

        x1_raw = batch.asip                                           # (B, C, H, W) with NaN
        y      = torch.cat((batch.input, batch.osisaf), dim=1)       # (B, 2C, H, W)
        B, C, H, W = x1_raw.shape

        valid = ~torch.isnan(x1_raw)                                 # (B, C, H, W)
        x1    = torch.nan_to_num(x1_raw, nan=0.0)                   # fill for interpolant

        t      = sample_uniform_time(x1)
        t_flat = t.view(B)
        noise  = torch.randn_like(x1)
        x_t    = t * x1 + (1.0 - t) * noise
        v_tgt  = x1 - noise

        v_pred = self.network(x_t, y, t_flat)

        log_s    = self.log_scale(t_flat).view(B, C, 1, 1).expand_as(v_pred)
        residual = (v_tgt - v_pred) / log_s.exp().clamp(min=1e-5)
        loss_map = neglogpdf(residual, log_s)

        loss = masked_average(loss_map, valid)

        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def on_train_epoch_end(self) -> None:
        if self._val_batch is None:
            return
        device = self.device
        dtype  = next(self.network.parameters()).dtype

        x_gt_raw = self._val_batch.asip.to(device=device, dtype=dtype)
        y = torch.cat((
            self._val_batch.input.to(device=device, dtype=dtype),
            self._val_batch.osisaf.to(device=device, dtype=dtype),
        ), dim=1)

        valid = ~torch.isnan(x_gt_raw)
        x_gt  = torch.nan_to_num(x_gt_raw, nan=0.0)

        ema_net = self.ema_model.module.to(device=device, dtype=dtype)
        ema_net.eval()

        noise = torch.randn_like(x_gt)
        x_hat, _ = heun_sample(ema_net, noise, y, n_steps=self.config.eval_n_steps)

        err = (x_hat.float() - x_gt.float()).pow(2)
        rmse = (err[valid].mean()).sqrt().item()
        print(f"[Epoch {self.current_epoch}]  Val RMSE (masked, EMA) = {rmse:.4f}")
        self.log("val_rmse_ema", rmse)

    def on_fit_start(self) -> None:
        try:
            dl = self.trainer.datamodule.val_dataloader()
            batch = next(iter(dl))
            if isinstance(batch, list):
                batch = batch[0]
            self._val_batch = batch
            print(f"Val batch cached: asip={batch.asip.shape}")
        except Exception as e:
            print(f"Could not cache val batch: {e}")

print("LitFlowMatchingSIC defined")

## Training

In [ ]:
def is_valid_checkpoint(path: str) -> bool:
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            zf.testzip()
        return True
    except Exception:
        return False

def find_best_valid_checkpoint(ckpt_dir: str) -> Optional[str]:
    if not os.path.isdir(ckpt_dir):
        return None
    last = os.path.join(ckpt_dir, "last.ckpt")
    if os.path.exists(last) and is_valid_checkpoint(last):
        print(f"Valid checkpoint: {last}")
        return last
    all_ckpts = sorted(
        [p for p in glob.glob(os.path.join(ckpt_dir, "*.ckpt")) if "last" not in p],
        key=lambda p: float(p.split("train_loss=")[-1].replace(".ckpt", ""))
        if "train_loss=" in p else float("inf"),
    )
    for p in all_ckpts:
        if is_valid_checkpoint(p):
            print(f"Valid checkpoint: {p}")
            return p
    print("No valid checkpoint found. Starting from scratch.")
    return None

LOG_DIR     = "logs_FM_sic"
CKPT_DIR    = os.path.join(LOG_DIR, "checkpoints")
MODEL_PATH  = os.path.join(LOG_DIR, "best_model")

# RESET_TRAINING set by the parameters cell above
if RESET_TRAINING:
    import shutil
    for d in [CKPT_DIR, MODEL_PATH]:
        if os.path.exists(d):
            shutil.rmtree(d)
    resume_ckpt = None
    print("RESET_TRAINING=True -- starting from scratch")
else:
    resume_ckpt = find_best_valid_checkpoint(CKPT_DIR)

network = UNetFM(UNetFMConfig(channels=C, cond_channels=2*C))
lit_fm  = LitFlowMatchingSIC(
    network=network,
    config=LitFMConfig(lr_scheduler_iters=1000, eval_n_steps=20),
)

trainer = Trainer(enable_progress_bar=False, 
    accelerator="gpu",
    max_epochs=MAX_EPOCHS,
    accumulate_grad_batches=4,
    precision="16-mixed",
    log_every_n_steps=1,
    logger=TensorBoardLogger(".", name=LOG_DIR, version=""),
    callbacks=[
        LearningRateMonitor(logging_interval="step"),
        ModelCheckpoint(
            dirpath=CKPT_DIR,
            monitor="train_loss",
            save_top_k=3,
            save_last=True,
            filename="{epoch:03d}-{step}-{train_loss:.4f}",
        ),
    ],
)

seed_everything(42)
trainer.callbacks.append(EpochHeartbeat(every_n_epochs=1))
if SKIP_TRAINING:
    if resume_ckpt is None:
        raise RuntimeError(
            f"SKIP_TRAINING=True but no checkpoint found in {CKPT_DIR} -- "
            "run training first (submit_train.sbatch) before computing metrics."
        )
    print(f'[TRAINING] SKIP_TRAINING=True -- loading weights from {resume_ckpt} directly (trainer.fit() not called)', flush=True)
    _ckpt_state = torch.load(resume_ckpt, map_location='cpu')
    lit_fm.load_state_dict(_ckpt_state['state_dict'])
else:
    print(f'[TRAINING] resume_ckpt={resume_ckpt!r} | MAX_EPOCHS={MAX_EPOCHS}', flush=True)
    trainer.fit(lit_fm, datamodule, ckpt_path=resume_ckpt)

ema_net = lit_fm.ema_model.module
ema_net.save_pretrained(MODEL_PATH)
print(f"EMA model saved to: {MODEL_PATH}")

## Sampling & Evaluation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float32

MODEL_PATH = os.path.join("logs_FM_sic", "best_model")

ema_unet = UNetFM.from_pretrained(MODEL_PATH).eval().to(device=device, dtype=dtype)
print(f"EMA UNetFM loaded from {MODEL_PATH}")
print(f"  Params : {sum(p.numel() for p in ema_unet.parameters()):,}")

### Test Batch

In [ ]:
seed_everything(42)
batch = next(iter(datamodule.test_dataloader()))

x_gt_raw = batch.asip.to(device=device, dtype=dtype)     # (B, C, H, W) with NaN
y_cond   = torch.cat((
    batch.input.to(device=device, dtype=dtype),
    batch.osisaf.to(device=device, dtype=dtype),
), dim=1)                                                  # (B, 2C, H, W)

B, C, H, W = x_gt_raw.shape
valid_mask = ~torch.isnan(x_gt_raw)
x_gt = torch.nan_to_num(x_gt_raw, nan=0.0)

m_norm, s_norm = datamodule.norm_stats()
print(f"Test batch  asip:{tuple(x_gt_raw.shape)}  y_cond:{tuple(y_cond.shape)}")
print(f"NaN fraction in target: {(~valid_mask).float().mean()*100:.1f}%")

### Ensemble Generation (N_SAMPLES members)

In [ ]:
# N_SAMPLES set by the parameters cell above
N_STEPS   = 20

samples = []

with torch.no_grad():
    for s in range(N_SAMPLES):
        noise = torch.randn_like(x_gt)
        x_pred, _ = heun_sample(ema_unet, noise, y_cond, n_steps=N_STEPS)
        samples.append(x_pred.cpu())
        print(f"  member {s+1}/{N_SAMPLES} done", end="\r")

samples = torch.stack(samples, dim=0)   # (N, B, C, H, W)
print(f"\nEnsemble: {tuple(samples.shape)}")

ens_mean = samples.mean(0)
ens_std  = samples.std(0)

_b = 0
ens_mean_1 = ens_mean[_b]   # (C, H, W)
ens_std_1  = ens_std[_b]
ensemble_1 = samples[:, _b]
gt_valid_1 = valid_mask[_b].cpu()

### Publication Figures — SIC comparison & uncertainty

In [ ]:
from matplotlib.gridspec import GridSpec

FIG_TAG = 'FM_sic'
FIG_DIR = os.path.join('figures', FIG_TAG)
os.makedirs(FIG_DIR, exist_ok=True)

N_SHOW = min(C, 5)

asip_phys  = batch.asip[_b, :N_SHOW].float().cpu().numpy() * s_norm + m_norm
input_phys = batch.input[_b, :N_SHOW].float().cpu().numpy() * s_norm + m_norm
osi_phys   = batch.osisaf[_b, :N_SHOW].float().cpu().numpy() * s_norm + m_norm
mean_phys  = ens_mean_1[:N_SHOW].float().cpu().numpy() * s_norm + m_norm
std_phys   = ens_std_1[:N_SHOW].float().cpu().numpy() * s_norm
mbr0_phys  = ensemble_1[0, :N_SHOW].float().cpu().numpy() * s_norm + m_norm

# Valid-pixel mask: keep only pixels where ASIP has >= 1 valid obs over
# the window (the dataset's land_mask and lat/lon are both unreliable,
# so derive validity directly from ASIP coverage instead of geolocation).
is_land = ~np.isfinite(asip_phys).any(axis=0)
print(f'Masked-out fraction (no ASIP obs in window): {is_land.mean()*100:.1f}%')
for _arr in (asip_phys, input_phys, osi_phys, mean_phys, std_phys, mbr0_phys):
    _arr[:, is_land] = np.nan

# SIC is a physical fraction in [0,1] (norm_stats=[0,1], no more /100 rescale).
SIC_MAX = 1.0

cmap_sic = plt.cm.Blues_r.copy(); cmap_sic.set_bad('lightgray')
cmap_std = plt.cm.Reds.copy();   cmap_std.set_bad('lightgray')

def _save_strip(data, filename, vmin, vmax, cmap, ncols=None):
    nc = ncols or data.shape[0]
    FW, FH, CB_H = 2.0, 2.0, 0.28
    fig_w = nc * FW
    fig_h = FH + CB_H + 0.06
    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = GridSpec(2, nc, left=0.01, right=0.99, top=0.99, bottom=0.01,
                  height_ratios=[FH, CB_H], hspace=0.06, wspace=0.03)
    for c in range(nc):
        ax = fig.add_subplot(gs[0, c])
        ax.imshow(data[c], origin='lower', cmap=cmap,
                  vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.axis('off')
    ax_cb = fig.add_subplot(gs[1, :])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=ax_cb, orientation='horizontal')
    cb.ax.tick_params(labelsize=9)
    fpath = os.path.join(FIG_DIR, f'{filename}.png')
    fig.savefig(fpath, dpi=200, bbox_inches='tight')
    print(f'  Saved: {fpath}')
    plt.show()

vmax_s = float(np.nanpercentile(std_phys, 99))

for data, fname, vmin, vmax, cmap in [
    (asip_phys,  f'{FIG_TAG}_asip_gt',   0, SIC_MAX, cmap_sic),
    (input_phys, f'{FIG_TAG}_input',     0, SIC_MAX, cmap_sic),
    (osi_phys,   f'{FIG_TAG}_osisaf',    0, SIC_MAX, cmap_sic),
    (mean_phys,  f'{FIG_TAG}_fm_mean',   0, SIC_MAX, cmap_sic),
    (std_phys,   f'{FIG_TAG}_fm_spread', 0, vmax_s, cmap_std),
    (mbr0_phys,  f'{FIG_TAG}_member0',   0, SIC_MAX, cmap_sic),
]:
    _save_strip(data, fname, vmin, vmax, cmap, ncols=N_SHOW)

### ODE Transport Process — Heun trajectory

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

T_CH   = C // 2
N_SHOW = 10

N_VIZ_STEPS = 20
torch.manual_seed(42)
noise_traj = torch.randn(1, C, H, W, device=device, dtype=dtype)
y_traj     = y_cond[0:1]

with torch.no_grad():
    _, trajectory = heun_sample(ema_unet, noise_traj, y_traj, n_steps=N_VIZ_STEPS)

transport_n = torch.stack(trajectory, dim=0)[:, 0, T_CH, :, :].float().cpu().numpy()
transport_p = transport_n * s_norm + m_norm

nfr = transport_p.shape[0]
stride = max(1, nfr // N_SHOW)
idx_show = list(range(0, nfr, stride))
if (nfr - 1) not in idx_show:
    idx_show.append(nfr - 1)
frames = transport_p[idx_show]
nf = len(frames)

# SIC is a physical fraction in [0,1] (norm_stats=[0,1], no more /100 rescale).
SIC_MAX = 1.0

FW, FH   = 2.4, 2.4
ARR_H    = 0.28
GAP_1    = 0.14
CB_H     = 0.26
GAP_2    = 0.16

fig_w = nf * FW
fig_h = ARR_H + GAP_1 + CB_H + GAP_2 + FH

arr_y_f   = (ARR_H / 2) / fig_h
cb_bot_f  = (ARR_H + GAP_1) / fig_h
cb_h_f    = CB_H / fig_h
img_bot_f = (ARR_H + GAP_1 + CB_H + GAP_2) / fig_h

fig = plt.figure(figsize=(fig_w, fig_h))
gs = GridSpec(1, nf, left=0.01, right=0.99, top=0.99, bottom=img_bot_f,
              hspace=0, wspace=0.02)

cmap_t = plt.cm.Blues_r
for i in range(nf):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(frames[i], origin='lower', cmap=cmap_t,
              vmin=0, vmax=SIC_MAX, interpolation='nearest')
    ax.axis('off')

cb_ax = fig.add_axes([0.15, cb_bot_f, 0.70, cb_h_f])
sm = plt.cm.ScalarMappable(cmap=cmap_t, norm=plt.Normalize(0, SIC_MAX))
sm.set_array([])
cb = fig.colorbar(sm, cax=cb_ax, orientation='horizontal')
cb.set_label('SIC', fontsize=24, fontweight='bold')
cb.ax.tick_params(labelsize=20)

fig.add_artist(mpatches.FancyArrowPatch(
    (0.04, 1+arr_y_f), (0.96, 1+arr_y_f),
    transform=fig.transFigure, arrowstyle='-|>',
    color='black', mutation_scale=32, linewidth=1.5,
))
fig.text(0.02, 1+arr_y_f, 'noise', va='bottom', ha='left',  fontsize=24, color='steelblue')
fig.text(0.98, 1+arr_y_f, 'clean', va='bottom', ha='right', fontsize=24, color='tomato')
fig.text(0.50, 1+arr_y_f, 'ODE time (Heun)', va='bottom', ha='center', fontsize=24)

fpath = os.path.join(FIG_DIR, f'{FIG_TAG}_transport.png')
fig.savefig(fpath, dpi=200, bbox_inches='tight')
print(f'  Saved: {fpath}')
plt.show()

## Metrics — FM vs GT (masked)

Metrics computed only on non-NaN pixels of the ASIP target.

| Metric | Description |
|---|---|
| **Score** | $1 - \text{RMSE}/\sigma_{\text{GT}}$ |
| **RMSE** | Root Mean Square Error (masked) |
| **CRPS** | Continuous Ranked Probability Score at valid pixels |
| **lambda_x** | Resolved scale (px) at spectral score = 0.5 |

In [ ]:
import sys
sys.path.append('../..')
from spectral_utils import radial_psd_2d, psd_spectral_score, resolved_scale

DX = 1.0   # pixel spacing (px)

# Marginal Ice Zone (MIZ) bounds -- standard glaciology convention (15-85%
# concentration). Pointwise metrics (Score/RMSE/sigma/CRPS) computed over the
# WHOLE domain are dominated by the huge near-constant open-water/full-ice
# background -- restricting to the MIZ targets the only part of the field
# with real reconstruction difficulty. lambda_x (spectral resolved scale)
# still needs the FULL 2D field (PSD/FFT can't be restricted to a scattered
# pixel mask), so it is computed per (sample, timestep) on the whole domain
# and then aggregated like the other metrics, not restricted itself.
MIZ_LO, MIZ_HI = 0.15, 0.85

# ── Full test-set evaluation ─────────────────────────────────────────────
# Loops over EVERY batch of datamodule.test_dataloader() (not just the first
# one used for the illustrative figures above) and EVERY timestep of the
# assimilation window (not just T_EVAL=C//2), regenerating a fresh ensemble
# each time. This is the expensive part (N_SAMPLES x n_test_batches Heun
# integrations) -- N_SAMPLES was reduced from 50 to 20 in methods.yaml
# specifically to keep a full-test-set pass affordable.
_all = {'score': [], 'rmse': [], 'sigma_gt': [], 'sigma_pred': [], 'crps': [], 'lambda_x': []}
_n_pairs = 0

seed_everything(42)
for _tb in datamodule.test_dataloader():
    _x_gt_b_raw = _tb.asip.to(device=device, dtype=dtype)         # (B, C, H, W) NaN ok
    _y_b = torch.cat((
        _tb.input.to(device=device, dtype=dtype),
        _tb.osisaf.to(device=device, dtype=dtype),
    ), dim=1)
    _Bb, _Cb, _Hb, _Wb = _x_gt_b_raw.shape
    _valid_b = ~torch.isnan(_x_gt_b_raw)

    with torch.no_grad():
        _members = []
        for _ in range(N_SAMPLES):
            _noise = torch.randn_like(_x_gt_b_raw)
            _x_pred, _ = heun_sample(ema_unet, _noise, _y_b, n_steps=N_STEPS)
            _members.append(_x_pred.cpu())
    _ens_b = torch.stack(_members, dim=0)         # (N, B, C, H, W) normalised
    _ens_mean_b = _ens_b.mean(0)                   # (B, C, H, W)

    for _bi in range(_Bb):
        for _t in range(_Cb):
            _valid_t = _valid_b[_bi, _t].cpu().numpy()
            if _valid_t.sum() < 10:
                continue
            _gt_p   = _x_gt_b_raw[_bi, _t].float().cpu().numpy() * s_norm + m_norm
            _mean_p = _ens_mean_b[_bi, _t].float().numpy()        * s_norm + m_norm
            _ens_p  = _ens_b[:, _bi, _t].float().numpy()          * s_norm + m_norm
            _miz_t  = _valid_t & (_gt_p > MIZ_LO) & (_gt_p < MIZ_HI)
            if _miz_t.sum() < 5:
                continue

            _gt_v, _pred_v = _gt_p[_miz_t], _mean_p[_miz_t]
            _sigma = float(np.std(_gt_v))
            if _sigma <= 0:
                continue
            _rmse  = float(np.sqrt(np.mean((_pred_v - _gt_v) ** 2)))
            _score = 1.0 - _rmse / _sigma
            _sigma_pred = float(np.std(_pred_v))

            _gt_f   = np.nan_to_num(_gt_p,   nan=0.0)
            _pred_f = np.nan_to_num(_mean_p, nan=0.0)
            _wl, _, _, _spec = psd_spectral_score(_pred_f, _gt_f, dx=DX)
            _lam = resolved_scale(_wl, _spec, threshold=0.5)

            _crps_val = np.nan
            if HAS_PROPERSCORING:
                _obs_ij = np.argwhere(_miz_t)
                if len(_obs_ij) > 0:
                    _stride = max(1, len(_obs_ij) // 50)   # subsample -- CRPS is O(n) Python loop
                    _crps_val = float(np.mean([
                        crps_ensemble(float(_gt_p[i, j]), _ens_p[:, i, j])
                        for i, j in _obs_ij[::_stride]
                    ]))

            _all['score'].append(_score)
            _all['rmse'].append(_rmse)
            _all['sigma_gt'].append(_sigma)
            _all['sigma_pred'].append(_sigma_pred)
            _all['crps'].append(_crps_val)
            _all['lambda_x'].append(_lam)
            _n_pairs += 1

print(f"Evaluated {_n_pairs} (test sample, timestep) pairs with MIZ coverage across the full test set")

def _agg(vals):
    a = np.asarray(vals, dtype=float)
    a = a[~np.isnan(a)]
    return (float(np.mean(a)), float(np.std(a))) if len(a) else (np.nan, np.nan)

_score_m, _score_s = _agg(_all['score'])
_rmse_m, _rmse_s = _agg(_all['rmse'])
_sgt_m, _sgt_s = _agg(_all['sigma_gt'])
_spr_m, _spr_s = _agg(_all['sigma_pred'])
_crps_m, _crps_s = _agg(_all['crps'])
_lam_m, _lam_s = _agg(_all['lambda_x'])

row_fm = {
    'Method'    : 'FM (ens. mean, MIZ, full test set)',
    'Score'     : f'{_score_m:.3f} ± {_score_s:.3f}',
    'RMSE'      : f'{_rmse_m:.4f} ± {_rmse_s:.4f}',
    'sigma_GT'  : f'{_sgt_m:.4f} ± {_sgt_s:.4f}',
    'sigma_pred': f'{_spr_m:.4f} ± {_spr_s:.4f}',
    'CRPS'      : f'{_crps_m:.4f} ± {_crps_s:.4f}' if not np.isnan(_crps_m) else '--',
    'lambda_x'  : f'{_lam_m:.1f} ± {_lam_s:.1f}' if not np.isnan(_lam_m) else '?',
}
df_metrics = pd.DataFrame([row_fm]).set_index('Method')
df_metrics.columns = ['Score','RMSE','sigma_GT','sigma_pred','CRPS','lambda_x [px]']
print(f'\n## Metrics -- full test set, MIZ only ({MIZ_LO}-{MIZ_HI}), n={_n_pairs} (sample,t) pairs\n')

# ── Canonicalize columns for the cross-method LaTeX table (make_latex_table.py) ──
# Every notebook in the suite must expose the SAME column names (RMSE,
# lambda_x, CRPS) regardless of internal naming (unicode arrows/sigma vs
# plain ascii, [px]/[deg] unit suffixes) -- otherwise make_latex_table.py's
# column-union logic creates duplicate columns (e.g. both "RMSE" and
# "RMSE ↓") instead of one shared column per metric. Score/sigma_GT/sigma_pred
# are dropped (not part of the target table). "±" is replaced with the
# LaTeX-safe "$\pm$" so the aggregated .tex table compiles cleanly.
_col_map = {
    'RMSE': 'RMSE', 'RMSE ↓': 'RMSE', 'RMSE down': 'RMSE',
    'lambda_x': 'lambda_x', 'lambda_x [px]': 'lambda_x', 'lambda_x [deg]': 'lambda_x',
    'lambda_x px': 'lambda_x', 'lambda_x [km]': 'lambda_x',
    'λx [px]': 'lambda_x', 'λx [deg]': 'lambda_x', 'λx px': 'lambda_x', 'λx [km]': 'lambda_x',
    'CRPS': 'CRPS', 'CRPS ↓': 'CRPS', 'CRPS down': 'CRPS',
}
df_metrics = df_metrics.rename(columns=_col_map)
for _c in df_metrics.columns:
    df_metrics[_c] = df_metrics[_c].apply(lambda v: v.replace('±', '$\\pm$') if isinstance(v, str) else v)
_keep = [c for c in ['RMSE', 'lambda_x', 'CRPS'] if c in df_metrics.columns]
df_metrics = df_metrics[_keep]

display(df_metrics)

# ── Illustrative PSD plot (single example, T_EVAL=C//2 of the first test
# batch, same one used by the figures above) -- qualitative check only, NOT
# the quantitative table (that's df_metrics above, full test set now).
T_EVAL = C // 2
gt_n_ex   = batch.asip[_b, T_EVAL].float().cpu().numpy()
gt_p_ex   = gt_n_ex * s_norm + m_norm
mean_n_ex = ens_mean_1[T_EVAL].float().cpu().numpy()
mean_p_ex = mean_n_ex * s_norm + m_norm
gt_filled_ex   = np.nan_to_num(gt_p_ex, nan=0.0)
mean_filled_ex = np.nan_to_num(mean_p_ex, nan=0.0)

wl_fm, _, _, spec_fm = psd_spectral_score(mean_filled_ex, gt_filled_ex, dx=DX)
_, psd_gt_r, psd_err_fm, _ = psd_spectral_score(mean_filled_ex, gt_filled_ex, dx=DX)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
valid_s = np.isfinite(wl_fm) & np.isfinite(psd_gt_r) & (psd_gt_r > 0)
ax.semilogy(wl_fm[valid_s], psd_gt_r[valid_s],  'k-',   lw=2,   label='GT')
valid_e = np.isfinite(wl_fm) & np.isfinite(psd_err_fm) & (psd_err_fm > 0)
ax.semilogy(wl_fm[valid_e], psd_err_fm[valid_e], 'C0--', lw=1.5, label='FM error')
ax.set_xlabel('Wavelength [px]')
ax.set_ylabel('PSD')
ax.set_title(f'PSD -- GT & error (illustrative, t={T_EVAL})')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
lambda_x_fm = resolved_scale(wl_fm, spec_fm)
lbl = f'FM (lambda_x={lambda_x_fm:.1f} px)' if not np.isnan(lambda_x_fm) else 'FM'
valid_sc = np.isfinite(wl_fm) & np.isfinite(spec_fm)
ax.plot(wl_fm[valid_sc], spec_fm[valid_sc], 'C0-', lw=2, label=lbl)
ax.axhline(0.5, color='gray', lw=1.2, ls='--', label='threshold 0.5')
if not np.isnan(lambda_x_fm):
    ax.axvline(lambda_x_fm, color='C0', lw=1, ls=':')
ax.set_xlabel('Wavelength [px]')
ax.set_ylabel('Spectral score')
ax.set_title('Score PSD = 1 - PSD(err) / PSD(GT) (illustrative)')
ax.set_ylim(-0.3, 1.05)
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- metrics serialization (patch_notebook_params.py) ---
import os
os.makedirs(os.path.dirname(METRICS_CSV) or '.', exist_ok=True)
_df_out = df_metrics.reset_index() if df_metrics.index.name == 'Method' else df_metrics
_df_out.to_csv(METRICS_CSV, index=False)
print(f'Metrics written to {METRICS_CSV}')
